# Notebook 1: Flood model training (Sen1Floods11)

Run the cells **from top to bottom, one at a time**. Each cell has a title comment and tells you what to expect.

* Cells 1-5 are quick checks and the Otsu baseline (E0). Do these on Friday.
* Cell 8 is the overfit test. Cell 9 is the first real training run (E1, seed 1): **note how long it takes**.
* Cells 10-14 are the long runs. They can be re-run safely: finished runs are skipped, interrupted runs resume.
* Cells 15-19 make tables, figures and the model file used by the app.

Set `QUICK = True` only for testing the notebook (tiny, fast runs). Leave it `False` for real runs.

In [ ]:
# CELL 1: setup and paths
import os, json, time, re, random
from pathlib import Path
import numpy as np, pandas as pd
import rasterio
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
from tqdm.auto import tqdm

ROOT = Path(os.environ.get("FLOOD_ROOT", str(Path.cwd() / "Data" / "sen1floods11_v1.1")))
WORK = Path(os.environ.get("FLOOD_WORK", str(ROOT.parents[1] / "work")))   # results go to Flood_Watch\work
QUICK = os.environ.get("FLOOD_QUICK", "0") == "1"                          # leave as 0 for real runs
for d in ("results", "models", "figures"):
    (WORK / d).mkdir(parents=True, exist_ok=True)

DEV = "cuda" if torch.cuda.is_available() else "cpu"
AMP = DEV == "cuda"
torch.backends.cudnn.benchmark = True
print("ROOT:", ROOT, "| exists:", ROOT.exists())
print("WORK:", WORK)
print("device:", DEV, torch.cuda.get_device_name(0) if AMP else "(CPU only: training will be slow)")

In [ ]:
# CELL 2: find the chips and read the official splits
D = ROOT / "data"
def index(folder):
    return {p.stem.rsplit("_", 1)[0]: p for p in (D / folder).glob("*.tif")}

S1, LAB, OTS, JRC = (index(f) for f in ("S1GRDHand", "LabelHand", "S1OtsuLabelHand", "JRCWaterHand"))
print("radar:", len(S1), "| labels:", len(LAB), "| otsu masks:", len(OTS), "| permanent water:", len(JRC))
ALL = sorted(set(S1) & set(LAB))
print("chips with radar + label:", len(ALL), "(expect 446)")

def read_split(name):
    f = next((ROOT / "splits").glob(f"flood_{name}_data*"))
    ids = []
    for line in f.read_text().splitlines():
        m = re.match(r"\s*([A-Za-z][A-Za-z\-]*_\d+)", line)
        if m:
            ids.append(m.group(1))
    return ids

SPLIT = {n: [c for c in read_split(n) if c in S1 and c in LAB] for n in ("train", "valid", "test", "bolivia")}
TRAIN, VALID, TEST, BOL = SPLIT["train"], SPLIT["valid"], SPLIT["test"], SPLIT["bolivia"]
INDIA_ALL = [c for c in ALL if c.startswith("India")]
TEST_INDIA = [c for c in TEST if c.startswith("India")]
rows = [(n, len(v), sum(c.startswith("India") for c in v)) for n, v in SPLIT.items()]
print(pd.DataFrame(rows, columns=["split", "chips", "India chips"]).to_string(index=False))
print("India chips in the whole dataset:", len(INDIA_ALL))
print("chips in more than one of train/valid/test (should be 0):",
      len((set(TRAIN) & set(VALID)) | (set(TRAIN) & set(TEST)) | (set(VALID) & set(TEST))))

In [ ]:
# CELL 3: look at one chip (sanity check). You should see radar images, a label and an Otsu mask.
def read_tif(path, band=None):
    with rasterio.open(path) as src:
        return src.read() if band is None else src.read(band)

chip = INDIA_ALL[0] if INDIA_ALL else ALL[0]
s1 = read_tif(S1[chip]); lab = read_tif(LAB[chip], 1)
print(chip, "| radar shape:", s1.shape, s1.dtype, "| dB range:", float(np.nanmin(s1)), float(np.nanmax(s1)),
      "| label values:", np.unique(lab))
panels = [(s1[0], "VV (dB)"), (s1[1], "VH (dB)"), (lab, "Hand label (-1/0/1)")]
if chip in OTS: panels.append((read_tif(OTS[chip], 1), "Otsu mask"))
if chip in JRC: panels.append((read_tif(JRC[chip], 1), "Permanent water (JRC)"))
fig, ax = plt.subplots(1, len(panels), figsize=(4 * len(panels), 4))
for a, (img, t) in zip(np.atleast_1d(ax), panels):
    a.imshow(img, cmap="gray"); a.set_title(t); a.axis("off")
plt.show()

In [ ]:
# CELL 4: load every chip into memory once (about 0.5 GB). Radar is clipped to -30..0 dB and scaled to 0..1.
def fit512(a, fill):
    out = np.full(a.shape[:-2] + (512, 512), fill, a.dtype)
    h, w = min(a.shape[-2], 512), min(a.shape[-1], 512)
    out[..., :h, :w] = a[..., :h, :w]
    return out

def norm_s1(x):
    x = np.nan_to_num(x.astype(np.float32), nan=-30.0, posinf=0.0, neginf=-30.0)
    return (np.clip(x, -30.0, 0.0) + 30.0) / 30.0

CACHE = {}
for c in tqdm(ALL, desc="loading chips"):
    x = fit512(norm_s1(read_tif(S1[c])[:2]), 0.0).astype(np.float16)
    y = fit512(read_tif(LAB[c], 1).astype(np.int8), -1)
    CACHE[c] = (x, y)
print("cached", len(CACHE), "chips | RAM (GB):", round(sum(v[0].nbytes + v[1].nbytes for v in CACHE.values()) / 1e9, 2))

In [ ]:
# CELL 5: metrics and the Otsu baseline (E0). No training needed.
from skimage.filters import threshold_otsu

def confusion(pred, y):                       # pred: bool (N,H,W); y: int8 (N,H,W) with -1 = ignore
    tp = ((pred == 1) & (y == 1)).sum(axis=(1, 2))
    fp = ((pred == 1) & (y == 0)).sum(axis=(1, 2))
    fn = ((pred == 0) & (y == 1)).sum(axis=(1, 2))
    return tp, fp, fn

def summarize(tp, fp, fn):
    tp, fp, fn = float(np.sum(tp)), float(np.sum(fp)), float(np.sum(fn))
    prec, rec = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    return dict(iou=tp / max(tp + fp + fn, 1), f1=2 * prec * rec / max(prec + rec, 1e-9), precision=prec, recall=rec)

def save_metrics(exp, seed, sets, extra=None):
    out = WORK / "results" / exp / f"seed{seed}"; out.mkdir(parents=True, exist_ok=True)
    (out / "metrics.json").write_text(json.dumps({"sets": sets, **(extra or {})}))
    (out / "DONE").write_text("ok")

TEST_SETS = {"test": TEST, "bolivia": BOL, "test_india": TEST_INDIA}
for mode in ("provided", "per_chip"):
    sets = {}
    for name, ids in TEST_SETS.items():
        if not ids: continue
        Y = np.stack([CACHE[c][1] for c in ids])
        if mode == "provided":      # the ready-made Otsu masks that ship with the dataset
            pred = np.stack([fit512(read_tif(OTS[c], 1).astype(np.int8), 0) == 1 for c in ids])
        else:                       # our own Otsu threshold on the VH band, one per chip
            pred = np.stack([CACHE[c][0][1].astype(np.float32) < threshold_otsu(CACHE[c][0][1].astype(np.float32)) for c in ids])
        tp, fp, fn = confusion(pred, Y)
        sets[name] = dict(metrics=summarize(tp, fp, fn), chips=ids, tp=tp.tolist(), fp=fp.tolist(), fn=fn.tolist())
    save_metrics(f"e0_otsu_{mode}", 0, sets)
    print("E0 Otsu", mode, {k: round(v["metrics"]["iou"], 3) for k, v in sets.items()}, "(water IoU)")

In [ ]:
# CELL 6: model, loss, augmentation, evaluation
def build_model(encoder="resnet34"):
    return smp.Unet(encoder, encoder_weights=None, in_channels=2, classes=1)   # no pretrained weights: nothing to download

def loss_fn(logits, y, kind="bce_dice"):
    logits = logits[:, 0]; m = (y != -1); yf = (y == 1).float()
    bce = F.binary_cross_entropy_with_logits(logits[m], yf[m]) if m.any() else logits.sum() * 0
    if kind == "bce":
        return bce
    p = torch.sigmoid(logits) * m; t = yf * m
    return bce + 1 - (2 * (p * t).sum() + 1) / (p.sum() + t.sum() + 1)

def get_crop(c, crop, rng):
    x, y = CACHE[c]
    i, j = rng.integers(0, 512 - crop + 1), rng.integers(0, 512 - crop + 1)
    xc, yc = x[:, i:i + crop, j:j + crop], y[i:i + crop, j:j + crop]
    k = int(rng.integers(0, 4))
    xc, yc = np.rot90(xc, k, axes=(1, 2)), np.rot90(yc, k, axes=(0, 1))
    if rng.random() < 0.5:
        xc, yc = xc[:, :, ::-1], yc[:, ::-1]
    return np.ascontiguousarray(xc), np.ascontiguousarray(yc)

@torch.no_grad()
def predict_probs(model, ids, bs=8):
    model.eval(); out = []
    for k in range(0, len(ids), bs):
        x = torch.from_numpy(np.stack([CACHE[c][0] for c in ids[k:k + bs]])).to(DEV).float()
        with torch.autocast(DEV, dtype=torch.float16, enabled=AMP):
            p = torch.sigmoid(model(x).float())[:, 0]
        out.append(p.cpu().numpy())
    return np.concatenate(out)

def evaluate(model, ids, thr=0.5):
    P = predict_probs(model, ids)
    Y = np.stack([CACHE[c][1] for c in ids])
    tp, fp, fn = confusion(P >= thr, Y)
    return summarize(tp, fp, fn), (tp, fp, fn), P

CFG = dict(
    encoder="resnet34",
    loss="bce_dice",
    lr=1e-4,
    wd=1e-4,
    bs=16,
    crop=256,
    epochs=60,
    crops_per_chip=4,
    patience=15,
)

if QUICK:
    CFG.update(
        encoder="resnet18",
        bs=4,
        crop=128,
        epochs=2,
        crops_per_chip=1,
        patience=2,
    )

print(CFG)
if QUICK:
    CFG.update(encoder="resnet18", bs=4, crop=128, epochs=2, crops_per_chip=1, patience=2)
print(CFG)

In [ ]:
# CELL 7: Train a model, save a checkpoint each epoch, and resume safely.

def run_train(exp, seed, train_ids, val_ids, cfg, keep_weights=True):
    out = WORK / "results" / exp / f"seed{seed}"
    out.mkdir(parents=True, exist_ok=True)

    summary_file = out / "summary.json"
    if summary_file.exists() and json.loads(summary_file.read_text()).get("finished"):
        print(f"{exp} seed{seed} is already finished.")
        return out

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model = build_model(cfg["encoder"]).to(DEV)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["wd"],
    )
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(
        opt,
        T_max=cfg["epochs"],
    )

    try:
        scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=AMP)

    start_epoch = 0
    best_iou = -1.0
    epochs_without_improvement = 0
    checkpoint_file = out / "last.pt"

    if checkpoint_file.exists():
        checkpoint = torch.load(checkpoint_file, map_location=DEV)

        model.load_state_dict(checkpoint["model"])
        opt.load_state_dict(checkpoint["opt"])
        sched.load_state_dict(checkpoint["sched"])

        # GradScaler state is empty when the checkpoint was saved on CPU.
        saved_scaler = checkpoint.get("scaler", {})
        if AMP and scaler.is_enabled() and saved_scaler:
            scaler.load_state_dict(saved_scaler)

        start_epoch = checkpoint["epoch"] + 1
        best_iou = checkpoint["best"]
        epochs_without_improvement = checkpoint["bad"]

        # Apply the learning rate from the current config after resuming.
        for group in opt.param_groups:
            group["lr"] = cfg["lr"]
        sched.base_lrs = [cfg["lr"] for _ in opt.param_groups]
        sched._last_lr = [cfg["lr"] for _ in opt.param_groups]

        print(f"Resuming {exp} seed{seed} from epoch {start_epoch}")

    log_file = out / "log.csv"
    if not log_file.exists():
        log_file.write_text("epoch,train_loss,val_iou,lr,seconds\n")

    last_epoch = start_epoch - 1

    for epoch in range(start_epoch, cfg["epochs"]):
        epoch_start = time.time()
        model.train()

        rng = np.random.default_rng(seed * 1000 + epoch)
        order = np.tile(np.array(train_ids), cfg["crops_per_chip"])
        rng.shuffle(order)

        batch_losses = []

        for k in range(0, len(order), cfg["bs"]):
            chips = order[k:k + cfg["bs"]]

            # Skip a final one-chip batch; this model's BatchNorm needs >1 item.
            if len(chips) < 2:
                continue

            batch = [get_crop(chip, cfg["crop"], rng) for chip in chips]
            x = torch.from_numpy(
                np.stack([item[0] for item in batch])
            ).to(DEV, dtype=torch.float32)
            y = torch.from_numpy(
                np.stack([item[1] for item in batch])
            ).to(DEV, dtype=torch.long)

            opt.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=DEV,
                dtype=torch.float16,
                enabled=AMP,
            ):
                logits = model(x)

            loss = loss_fn(logits.float(), y, cfg["loss"])
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()

            batch_losses.append(float(loss.item()))

        sched.step()

        val_iou = evaluate(model, val_ids)[0]["iou"]
        mean_train_loss = (
            float(np.mean(batch_losses)) if batch_losses else float("nan")
        )

        if val_iou > best_iou:
            best_iou = val_iou
            epochs_without_improvement = 0
            torch.save(model.state_dict(), out / "best.pt")
        else:
            epochs_without_improvement += 1

        torch.save(
            {
                "model": model.state_dict(),
                "opt": opt.state_dict(),
                "sched": sched.state_dict(),
                "scaler": scaler.state_dict(),
                "epoch": epoch,
                "best": best_iou,
                "bad": epochs_without_improvement,
            },
            checkpoint_file,
        )

        elapsed = time.time() - epoch_start
        with log_file.open("a") as f:
            f.write(
                f"{epoch},{mean_train_loss:.5f},{val_iou:.5f},"
                f"{opt.param_groups[0]['lr']:.2e},{elapsed:.1f}\n"
            )

        last_epoch = epoch
        summary_file.write_text(
            json.dumps(
                {
                    "best_val_iou": best_iou,
                    "epochs_run": epoch + 1,
                    "finished": False,
                }
            )
        )

        print(
            f"{exp} seed{seed} | "
            f"epoch {epoch + 1}/{cfg['epochs']} | "
            f"loss {mean_train_loss:.3f} | "
            f"val IoU {val_iou:.3f} | "
            f"best IoU {best_iou:.3f} | "
            f"{elapsed:.0f}s"
        )

        if epochs_without_improvement >= cfg["patience"]:
            print("Early stopping: validation IoU did not improve.")
            break

    summary_file.write_text(
        json.dumps(
            {
                "best_val_iou": best_iou,
                "epochs_run": last_epoch + 1,
                "finished": True,
            }
        )
    )

    checkpoint_file.unlink(missing_ok=True)

    if not keep_weights:
        (out / "best.pt").unlink(missing_ok=True)

    return out

In [ ]:
# CELL 8: experiment runner (trains, tunes the threshold on validation, scores the test sets, writes DONE)
def tune_threshold(P, Y):
    best_t, best_s = 0.5, -1.0
    for t in np.arange(0.30, 0.701, 0.05):
        s = summarize(*confusion(P >= t, Y))["iou"]
        if s > best_s:
            best_t, best_s = float(t), s
    return best_t

def run_experiment(exp, seeds, train_ids, val_ids, eval_sets, cfg):
    for seed in seeds:
        out = run_train(exp, seed, train_ids, val_ids, cfg)
        if (out / "DONE").exists():
            continue
        model = build_model(cfg["encoder"]).to(DEV)
        model.load_state_dict(torch.load(out / "best.pt", map_location=DEV))
        Pv = evaluate(model, val_ids)[2]
        thr = tune_threshold(Pv, np.stack([CACHE[c][1] for c in val_ids]))
        sets = {}
        for name, ids in eval_sets.items():
            if not ids: continue
            m, (tp, fp, fn), _ = evaluate(model, ids, thr)
            sets[name] = dict(metrics=m, chips=ids, tp=tp.tolist(), fp=fp.tolist(), fn=fn.tolist())
        save_metrics(exp, seed, sets, {"threshold": thr, "cfg": cfg})
        print(f"==> {exp} seed{seed}: threshold {thr:.2f}", {k: round(v["metrics"]["iou"], 3) for k, v in sets.items()})

EVAL_STD = {"test": TEST, "bolivia": BOL, "test_india": TEST_INDIA}

In [ ]:
# CELL 9 (overfit test, ~2-5 min): train on 16 chips and score on the same 16 chips.
# PASS: val IoU rises clearly (above about 0.6). FAIL: stays near 0 -> something is wrong; tell me before continuing.
sub = TRAIN[:16]
ocfg = dict(CFG, epochs=2 if QUICK else 40, crops_per_chip=4, patience=100, lr=1e-3)
out = run_train("overfit_check", 1, sub, sub, ocfg, keep_weights=False)
print(json.loads((out / "summary.json").read_text()))

In [ ]:
# CELL 10 (E1, seed 1): the first real run. WRITE DOWN how long one epoch takes and how long the run takes.
t0 = time.time()
run_experiment("e1_baseline", [1], TRAIN, VALID, EVAL_STD, CFG)
print("minutes for this run:", round((time.time() - t0) / 60, 1))

In [ ]:
# CELL 11 (E1, seeds 2 and 3): leave this running; finished runs are skipped if you re-run the cell
run_experiment("e1_baseline", [2, 3], TRAIN, VALID, EVAL_STD, CFG)

In [ ]:
# CELL 12 (E3, leave-India-out): the model never sees any India chip. We then test on every India chip.
TRAIN_NI = [c for c in TRAIN if not c.startswith("India")]
VALID_NI = [c for c in VALID if not c.startswith("India")]
EVAL_E3 = {"india_all": INDIA_ALL, "test": TEST, "bolivia": BOL, "test_india": TEST_INDIA}
run_experiment("e3_leave_india_out", [1, 2, 3], TRAIN_NI, VALID_NI, EVAL_E3, CFG)

In [ ]:
# CELL 13 (optional E2a, loss ablation): BCE only, no Dice term. Skip this cell if you are short of time.
run_experiment("e2a_bce_only", [1, 2, 3], TRAIN, VALID, EVAL_STD, dict(CFG, loss="bce"))

In [ ]:
# CELL 14 (E4, hyperparameter search): each trial is a short run. Change N_TRIALS if it is too slow.
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
N_TRIALS = 8
study = optuna.create_study(direction="maximize", study_name="e4",
                            storage="sqlite:///" + (WORK / "results" / "optuna.db").as_posix(), load_if_exists=True)

def objective(trial):
    cfg = dict(CFG, lr=trial.suggest_float("lr", 5e-5, 3e-4, log=True),
               wd=trial.suggest_float("wd", 1e-6, 1e-2, log=True),
               bs=trial.suggest_categorical("bs", [8, 16]),
               crop=trial.suggest_categorical("crop", [192, 256, 384]),
               epochs=2 if QUICK else 30, patience=8)
    if QUICK:
        cfg.update(bs=4, crop=128)
    out = run_train(f"e4_trial{trial.number}", 1, TRAIN, VALID, cfg, keep_weights=False)
    return json.loads((out / "summary.json").read_text())["best_val_iou"]

remaining = N_TRIALS - len([t for t in study.trials if t.state.name == "COMPLETE"])
if remaining > 0:
    study.optimize(objective, n_trials=remaining)
print("best validation IoU:", study.best_value)
print("best parameters:", study.best_params)
(WORK / "results" / "best_params.json").write_text(json.dumps(study.best_params))

In [ ]:
# CELL 15 (E5, final runs): best settings from E4 (or the defaults if you skipped E4), 5 seeds. Test sets are scored once here.
bp_file = WORK / "results" / "best_params.json"
CFG_BEST = dict(CFG)
if bp_file.exists():
    CFG_BEST.update(json.loads(bp_file.read_text())); CFG_BEST.update(epochs=CFG["epochs"], patience=CFG["patience"])
    if QUICK:
        CFG_BEST.update(bs=4, crop=128)
print(CFG_BEST)
run_experiment("e5_final", [1, 2, 3], TRAIN, VALID, EVAL_STD, CFG_BEST)

In [ ]:
# CELL 16: results table (mean ± standard deviation across seeds). Saved as summary_table.csv
def load_all():
    rows = []
    for expdir in sorted((WORK / "results").iterdir()):
        if not expdir.is_dir() or expdir.name.startswith("e4_trial") or expdir.name == "overfit_check":
            continue
        per = {}
        for sd in sorted(expdir.glob("seed*")):
            mj = sd / "metrics.json"
            if mj.exists():
                for name, v in json.loads(mj.read_text())["sets"].items():
                    per.setdefault(name, []).append(v["metrics"])
        for name, lst in per.items():
            row = dict(experiment=expdir.name, eval_set=name, seeds=len(lst))
            for k in ("iou", "f1", "precision", "recall"):
                vals = [m[k] for m in lst]
                row[k] = f"{np.mean(vals):.3f} ± {np.std(vals):.3f}" if len(vals) > 1 else f"{vals[0]:.3f}"
            rows.append(row)
    return pd.DataFrame(rows)

TABLE = load_all()
display(TABLE)
TABLE.to_csv(WORK / "results" / "summary_table.csv", index=False)

In [ ]:
# CELL 17: 95% confidence intervals (bootstrap over test chips) for each experiment's test set
def bootstrap_ci(exp, name="test", n=2000, seed=0):
    tps, fps, fns = [], [], []
    for sd in sorted((WORK / "results" / exp).glob("seed*")):
        mj = sd / "metrics.json"
        if mj.exists() and name in json.loads(mj.read_text())["sets"]:
            s = json.loads(mj.read_text())["sets"][name]
            tps.append(s["tp"]); fps.append(s["fp"]); fns.append(s["fn"])
    if not tps: return None
    tp, fp, fn = (np.mean(a, axis=0) for a in (tps, fps, fns))        # average the seeds chip by chip
    rng = np.random.default_rng(seed); vals = []
    for _ in range(n):
        i = rng.integers(0, len(tp), len(tp))
        vals.append(tp[i].sum() / max(tp[i].sum() + fp[i].sum() + fn[i].sum(), 1))
    return float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

for exp in ("e1_baseline", "e3_leave_india_out", "e5_final"):
    for name in ("test", "india_all"):
        ci = bootstrap_ci(exp, name)
        if ci: print(f"{exp:22s} {name:10s} water IoU 95% CI: {ci[0]:.3f} to {ci[1]:.3f}")

In [ ]:
# CELL 18: figures for the paper (saved to the figures folder)
def best_seed(exp):
    best = (None, -1)
    for sd in (WORK / "results" / exp).glob("seed*"):
        if (sd / "summary.json").exists():
            v = json.loads((sd / "summary.json").read_text())["best_val_iou"]
            if v > best[1]: best = (sd, v)
    return best[0]

exp = "e5_final" if best_seed("e5_final") else "e1_baseline"
sd = best_seed(exp)
cfgm = json.loads((sd / "metrics.json").read_text())
model = build_model(cfgm["cfg"]["encoder"]).to(DEV); model.load_state_dict(torch.load(sd / "best.pt", map_location=DEV))
thr = cfgm["threshold"]

# training curves
fig, ax = plt.subplots(1, 2, figsize=(10, 3.5))
for s in sorted((WORK / "results" / exp).glob("seed*")):
    lg = pd.read_csv(s / "log.csv"); ax[0].plot(lg.epoch, lg.train_loss, label=s.name); ax[1].plot(lg.epoch, lg.val_iou, label=s.name)
ax[0].set_title("Training loss"); ax[1].set_title("Validation water IoU"); ax[0].legend(); plt.tight_layout()
plt.savefig(WORK / "figures" / "training_curves.png", dpi=200); plt.show()

# example predictions: worst 3, best 3 and India chips of the test set
ids = TEST
_, (tp, fp, fn), P = evaluate(model, ids, thr)
chip_iou = tp / np.maximum(tp + fp + fn, 1)
order = np.argsort(chip_iou)
pick = list(order[:3]) + list(order[-3:]) + [i for i, c in enumerate(ids) if c.startswith("India")][:3]
fig, ax = plt.subplots(len(pick), 3, figsize=(9, 3 * len(pick)))
for r, i in enumerate(pick):
    c = ids[i]
    ax[r, 0].imshow(CACHE[c][0][1].astype(np.float32), cmap="gray"); ax[r, 0].set_title(f"{c} VH")
    ax[r, 1].imshow(np.where(CACHE[c][1] == -1, 0.5, CACHE[c][1]), cmap="gray", vmin=0, vmax=1); ax[r, 1].set_title("Hand label")
    ax[r, 2].imshow(P[i] >= thr, cmap="gray"); ax[r, 2].set_title(f"Prediction (IoU {chip_iou[i]:.2f})")
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.savefig(WORK / "figures" / "example_masks.png", dpi=150); plt.show()

In [ ]:
# CELL 19: save the final model for the app, and write the helper module the app will use for uploads
meta = dict(encoder=cfgm["cfg"]["encoder"], threshold=thr, norm="clip -30..0 dB then (x+30)/30", source_run=str(sd.relative_to(WORK)))
sdict = torch.load(sd / "best.pt", map_location="cpu")
torch.save({k: (v.half() if v.is_floating_point() else v) for k, v in sdict.items()}, WORK / "models" / "unet_fp16.pt")
(WORK / "models" / "model_meta.json").write_text(json.dumps(meta))
print("saved:", WORK / "models" / "unet_fp16.pt", meta)

INFER_CODE = '''
import numpy as np, torch, rasterio
import segmentation_models_pytorch as smp

MAX_PIXELS = 36_000_000

def load_model(weights_path, meta, device=None):
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    m = smp.Unet(meta["encoder"], encoder_weights=None, in_channels=2, classes=1)
    sd = torch.load(weights_path, map_location="cpu")
    m.load_state_dict({k: (v.float() if v.is_floating_point() else v) for k, v in sd.items()})
    return m.to(device).eval(), device

def to_model_input(a):
    # a: (bands, H, W) radar array. Returns (2, H, W) float32 in 0..1 plus a note about units.
    if a.shape[0] < 2:
        raise ValueError("Need a GeoTIFF with at least 2 bands (VV and VH).")
    a = a[:2].astype(np.float32)
    finite = a[np.isfinite(a)]
    if finite.size == 0:
        raise ValueError("The image has no valid pixels.")
    note = "values treated as dB"
    if finite.min() >= 0:
        if finite.max() > 50:
            raise ValueError("Values look like 8-bit or optical imagery, not Sentinel-1 backscatter. Upload VV/VH radar.")
        a = 10.0 * np.log10(np.maximum(a, 1e-6)); note = "linear power converted to dB"
    a = np.nan_to_num(a, nan=-30.0, posinf=0.0, neginf=-30.0)
    return (np.clip(a, -30.0, 0.0) + 30.0) / 30.0, note

def pixel_area_km2(transform, crs, lat):
    rx, ry = abs(transform.a), abs(transform.e)
    if crs is not None and crs.is_geographic:
        return (rx * 111.32 * np.cos(np.radians(lat))) * (ry * 110.57)
    return rx * ry / 1e6

@torch.no_grad()
def predict_array(model, device, x, tile=512, overlap=64):
    # x: (2,H,W) in 0..1. Returns water probability (H,W) using overlapping tiles.
    _, H, W = x.shape
    step = tile - overlap
    prob = np.zeros((H, W), np.float32); wsum = np.zeros((H, W), np.float32)
    win = np.outer(np.hanning(tile) + 1e-3, np.hanning(tile) + 1e-3).astype(np.float32)
    for i in range(0, max(H - overlap, 1), step):
        for j in range(0, max(W - overlap, 1), step):
            i0, j0 = min(i, max(H - tile, 0)), min(j, max(W - tile, 0))
            patch = x[:, i0:i0 + tile, j0:j0 + tile]
            ph, pw = patch.shape[1:]
            pad = np.zeros((2, tile, tile), np.float32); pad[:, :ph, :pw] = patch
            t = torch.from_numpy(pad)[None].to(device)
            with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
                p = torch.sigmoid(model(t).float())[0, 0].cpu().numpy()
            prob[i0:i0 + ph, j0:j0 + pw] += p[:ph, :pw] * win[:ph, :pw]
            wsum[i0:i0 + ph, j0:j0 + pw] += win[:ph, :pw]
    return prob / np.maximum(wsum, 1e-6)

def predict_tif(path, model, device, thr=0.5, tile=512, overlap=64):
    with rasterio.open(path) as src:
        if src.width * src.height > MAX_PIXELS:
            raise ValueError("Image is too large. Crop it to under about 6000 x 6000 pixels.")
        a = src.read(); transform, crs, bounds = src.transform, src.crs, src.bounds
    x, note = to_model_input(a)
    valid = np.isfinite(a[:2]).all(axis=0)
    prob = predict_array(model, device, x, tile, overlap)
    mask = (prob >= thr) & valid
    lat = (bounds.top + bounds.bottom) / 2
    px_km2 = pixel_area_km2(transform, crs, lat)
    nvalid = int(valid.sum()); nwater = int(mask.sum())
    stats = dict(valid_pixels=nvalid, water_pixels=nwater,
                 percent_water=float(100.0 * nwater / max(nvalid, 1)), water_km2=float(nwater * px_km2),
                 pixel_area_km2=float(px_km2), note=note)
    return dict(prob=prob, mask=mask, stats=stats, transform=transform, crs=crs, bounds=bounds)

def severity(percent):
    return "minor" if percent < 5 else "moderate" if percent < 20 else "major" if percent < 40 else "severe"
'''
(WORK / "flood_infer.py").write_text(INFER_CODE.strip() + "\n")
print("wrote", WORK / "flood_infer.py")

In [ ]:
# CELL 20: test the upload path on a real test chip. It should agree closely with the direct evaluation.
import sys, importlib
sys.path.insert(0, str(WORK)); import flood_infer; importlib.reload(flood_infer)
m, dev = flood_infer.load_model(WORK / "models" / "unet_fp16.pt", meta)
c = TEST[0]
res = flood_infer.predict_tif(S1[c], m, dev, thr=meta["threshold"])
print(c, res["stats"])
y = CACHE[c][1]
tp, fp, fn = confusion(res["mask"][None, :512, :512], y[None, :res["mask"].shape[0], :res["mask"].shape[1]])
print("IoU of the upload pipeline on this chip:", round(float(tp[0] / max(tp[0] + fp[0] + fn[0], 1)), 3))
print("IoU of the direct evaluation on the same chip (should be nearly the same):", round(evaluate(model, [c], meta["threshold"])[0]["iou"], 3))
print("Severity:", flood_infer.severity(res["stats"]["percent_water"]))

In [ ]:
# EXTRA CELL A: save run details and library versions for the paper
import platform, sys, importlib.metadata as md
info = {"python": sys.version.split()[0], "platform": platform.platform(),
        "torch": torch.__version__, "cuda_build": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
        "smp": md.version("segmentation-models-pytorch"), "numpy": np.__version__,
        "rasterio": rasterio.__version__,
        "chips": {k: len(v) for k, v in SPLIT.items()}, "india_chips_total": len(INDIA_ALL)}
rows = []
for expdir in sorted((WORK / "results").iterdir()):
    if not expdir.is_dir() or expdir.name.startswith("e4_trial") or expdir.name == "overfit_check":
        continue
    for sd in sorted(expdir.glob("seed*")):
        mj, sj, lg = sd / "metrics.json", sd / "summary.json", sd / "log.csv"
        if not mj.exists():
            continue
        m = json.loads(mj.read_text()); s = json.loads(sj.read_text()) if sj.exists() else {}
        cfg = m.get("cfg", {})
        rows.append(dict(experiment=expdir.name, seed=sd.name, threshold=m.get("threshold"),
                         epochs_run=s.get("epochs_run"), best_val_iou=s.get("best_val_iou"),
                         train_minutes=round(pd.read_csv(lg)["seconds"].sum() / 60, 1) if lg.exists() else None,
                         encoder=cfg.get("encoder"), loss=cfg.get("loss"), lr=cfg.get("lr"),
                         wd=cfg.get("wd"), bs=cfg.get("bs"), crop=cfg.get("crop")))
RUNS = pd.DataFrame(rows)
RUNS.to_csv(WORK / "results" / "run_details.csv", index=False)
(WORK / "results" / "environment.json").write_text(json.dumps(info, indent=2))
display(RUNS); print(info)

In [ ]:
# EXTRA CELL B: paired bootstrap, U-Net (E5) minus Otsu (provided masks), on the same chips
def counts(exp, name):
    tps, fps, fns, chips = [], [], [], None
    for sd in sorted((WORK / "results" / exp).glob("seed*")):
        mj = sd / "metrics.json"
        if mj.exists() and name in json.loads(mj.read_text())["sets"]:
            s = json.loads(mj.read_text())["sets"][name]
            tps.append(s["tp"]); fps.append(s["fp"]); fns.append(s["fn"]); chips = s["chips"]
    return chips, np.mean(tps, 0), np.mean(fps, 0), np.mean(fns, 0)

def iou(tp, fp, fn):
    return tp.sum() / max(tp.sum() + fp.sum() + fn.sum(), 1)

for name in ("test", "bolivia", "test_india"):
    c1, a_tp, a_fp, a_fn = counts("e5_final", name)
    c0, b_tp, b_fp, b_fn = counts("e0_otsu_provided", name)
    assert c1 == c0
    rng = np.random.default_rng(0); diffs = []
    for _ in range(2000):
        i = rng.integers(0, len(c1), len(c1))
        diffs.append(iou(a_tp[i], a_fp[i], a_fn[i]) - iou(b_tp[i], b_fp[i], b_fn[i]))
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    verdict = "U-Net clearly better" if lo > 0 else "Otsu clearly better" if hi < 0 else "no clear difference"
    print(f"{name:11s} U-Net minus Otsu IoU: {np.mean(diffs):+.3f}  (95% CI {lo:+.3f} to {hi:+.3f})  -> {verdict}")